# Sweep TSM-10 / KS-10: official and retrained 16–16

Pagani-style dominant-output curves and red/yellow/green binary-probe maps. The source protocol is `reconstruction_figures.ipynb`; this companion preserves its 512-sample context and 512-sample autoregressive median rollout (eight native 64-step predictions). This differs from the direct 64-point decomposition recovery experiment.

Both checkpoints see exactly the same inputs. Default: ten backgrounds per family, ten components each, every integer frequency 2–250 Hz. Injection amplitude is **1.5 × the largest background component amplitude**, with no renormalization of the mixture. A coincident background frequency can reinforce or cancel the injected tone.


In [ ]:
from pathlib import Path
import sys, os
HERE = Path.cwd().resolve()
REPO = next(p for p in (HERE, *HERE.parents) if (p / 'chronos/bayesian/support_scripts/comparison_lib.py').exists())
BAYES = REPO / 'chronos/bayesian'
sys.path.insert(0, str(BAYES / "support_scripts"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from dataclasses import replace
from IPython.display import display
torch.set_num_threads(8)
MODE = os.environ.get('PATCHALIASING_COMPARISON_MODE', 'full')
if MODE not in ('full','smoke'): raise ValueError(MODE)
ROOT_OUTPUT = BAYES / '_run/comparison'
CSV_PATH = REPO / 'chronos/data/dataset/Dataset-SolarTechLab.csv'


In [ ]:
import complex_sweep_lib as sweep
CONFIG = sweep.SweepConfig()
MODELS = sweep.MODELS16
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if MODE == 'smoke':
    CONFIG = replace(CONFIG, mode='smoke', n_backgrounds=2, frequency_step=32,
                     n_phase_probe=2, n_phase_rollout=1, rollout_length=64)
    print('SMOKE: reduced grid/backgrounds and direct 64-point horizon; not a full result.')
display(pd.DataFrame([vars(CONFIG)]))


## Backgrounds, amplitude and validation

TSM uses the existing Light TSMixup frequency pool, ten components separated by at least 10 Hz and Dirichlet weights (α=1.5). KS draws the existing KernelSynth process, then reconstructs **only the ten largest Fourier terms within 2–250 Hz**. KS-10 is therefore a finite harmonic reconstruction, not the full Gaussian-process realization. Fourier terms need not be ten isolated local peaks.

Both reconstructed backgrounds are standardized to unit standard deviation before injection. Component amplitudes and cosine phases, seeds and trial metadata are saved. The same background bank is reused at every frequency and for both checkpoints.

The seven LL/L/LH/Mid/HL/H/HH probes use the existing band definitions and output-head representation. Frequency-grouped cross-validation holds all phases of a frequency together; PCA/scaling are fitted inside each training fold. This tests new frequencies conditional on the shared background bank. The local ±1 Hz row holds out entire backgrounds. Its gray cells were not evaluated. Accuracy 0.5 is chance for balanced classes; the original bands and available phase counts can yield class imbalance.


In [ ]:
RESULTS = {}
for family in ('tsmixup', 'kernelsynth'):
    outputs, directory = sweep.run_family(family, ROOT_OUTPUT, CONFIG, MODELS, DEVICE)
    RESULTS[family] = (outputs, directory)
    display(sweep.plot_family(family, outputs, directory))
    plt.close('all')
    for model, (probes, trials, summary) in outputs.items():
        print(family, model, directory)
        display(probes.groupby('task', dropna=False).apply(
            lambda g: np.average(g.accuracy, weights=g.n_test), include_groups=False).rename('accuracy').to_frame())
        display(summary)


## Interpretation and saved artifacts

Each red point is a trial's strongest non-DC output FFT component. The red line averages these frequencies and its band is ±1 standard deviation; the mean need not itself be a component present in an output. Flat outputs have no dominant frequency and remain missing, with valid/total counts in CSV. In the full run the 512-point FFT spacing is 1 Hz. In smoke the 64-point FFT spacing is 8 Hz.

Each family saves its own PNG, component/trial CSVs, binary fold results, dominant-frequency summaries and resumable prediction/representation blocks. Cache identity includes code, model revisions, configuration and background signals. `full` and `smoke` outputs are separate. Original pure-tone sweep notebooks are unchanged.
